# ✏️ SketchesToLife — reproducible Colab run

This notebook reproduces the project end to end on a **GPU runtime**
(*Runtime ▸ Change runtime type ▸ T4 GPU*):

1. clone the repo and install the pinned dependencies
2. run the environment check
3. preprocess the sketches into control maps
4. **reproduce the hero figure** with the exact E6 configs and seed (42)
5. *(optional)* regenerate every figure in `results/` with `scripts/reproduce.sh`, and
   compare its SHA-256 manifest with a previous run
6. *(optional)* launch the Gradio app with a public `share=True` link

Every step is idempotent, so it is safe to re-run cells after a runtime restart.
Determinism caveats are listed at the end of the notebook.

In [ ]:
#@title ⚙️ Settings
REPO_URL = "https://github.com/<your-username>/SketchesToLife.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
RUN_FULL_REPRODUCTION = False  #@param {type:"boolean"}
LAUNCH_GRADIO_APP = False  #@param {type:"boolean"}

REPO_DIR = "/content/SketchesToLife"
DRIVE_DIR = "/content/drive/MyDrive/sketch2image"

## 1 · Clone and install

In [ ]:
import os, subprocess, sys

if "<your-username>" in REPO_URL:
    raise ValueError("Set REPO_URL in the Settings cell to your GitHub repository first.")

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "⚠️ No GPU: switch the runtime to T4 GPU"

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="Commit %h  (%cd)  %s" --date=short

In [ ]:
# Pinned stack from requirements.txt (diffusers 0.27.2, transformers 4.40.2, ...).
# Colab's preinstalled torch is kept; the exact versions are recorded in step 2.
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

# requirements.txt pins numpy 1.26; if Colab had numpy 2.x already imported, the
# runtime must restart once to load the pinned version. This cell does it for you;
# afterwards, simply run the notebook again from the top (installs are then no-ops).
import importlib.metadata, numpy
installed = importlib.metadata.version("numpy")
if numpy.__version__ != installed:
    print(f"numpy {numpy.__version__} loaded but {installed} installed -> restarting runtime…")
    os.kill(os.getpid(), 9)
print("Dependencies ready. numpy", numpy.__version__)

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.environ["OUTPUT_DIR"] = DRIVE_DIR

## 2 · Environment check

In [ ]:
%cd {REPO_DIR}
!python scripts/check_env.py

## 3 · Preprocess the sketches

Each sketch is cleaned (illumination normalisation, adaptive threshold, speck removal)
and converted into the control map ControlNet receives. The grids show raw → cleaned → Canny.

In [ ]:
%cd {REPO_DIR}
!PYTHONPATH=src python -m sketch2image.preprocessing --input sketches --out outputs/conditions --modes clean,canny

from IPython.display import Image as IPImage, display
import glob
for grid in sorted(glob.glob("outputs/conditions/*_grid.png")):
    print(grid); display(IPImage(grid, width=900))

## 4 · Reproduce the hero figure

The hero figure pairs each sketch with one E6 (style preset) output. Its configuration is
fixed in code and YAML: SD 1.5 + ControlNet **scribble**, conditioning scale 1.0, CFG 7.5,
30 steps, DPM++ 2M Karras, 512 px, **seed 42**; presets photorealistic / concept_art /
watercolor. The cell below prints the config it runs, generates E6 into a fresh folder
(no reuse of earlier images), and rebuilds `results/hero_comparison.png`.

In [ ]:
%cd {REPO_DIR}
!cat configs/experiments/e6_styles.yaml
!rm -rf outputs/hero_repro
!PYTHONHASHSEED=0 CUBLAS_WORKSPACE_CONFIG=:4096:8 python scripts/run_experiments.py -e E6 --out-dir outputs/hero_repro/experiments --results-dir results

display(IPImage("results/hero_comparison.png", width=700))

In [ ]:
# Per-image record: seed, model IDs, runtime and peak VRAM are stored next to every output.
import json, pathlib
for meta in sorted(pathlib.Path("outputs/hero_repro/experiments/E6").glob("*/*_meta.json"))[:3]:
    m = json.loads(meta.read_text())
    print(f"{meta.name}: seed={m['seed']} preset={m['style_preset']} "
          f"runtime={m['runtime_sec']}s vram={m['peak_vram_mb']:.0f}MB mock={m['mock']}")

## 5 · *(optional)* Regenerate every figure + verify against a previous run

`scripts/reproduce.sh` wipes its scratch folder, reruns E1, E2 and E6 from the configs,
rebuilds all grids, the hero figure, `metrics.csv`, the plots and `ANALYSIS.md`, then writes
a SHA-256 manifest of every generated image and figure (`outputs/reproduce/checksums.sha256`).

If `SAVE_TO_DRIVE` is on, the manifest is kept on Drive **per GPU model**, and the next run on
the same GPU type is compared with it automatically.

In [ ]:
%cd {REPO_DIR}
if RUN_FULL_REPRODUCTION:
    !bash scripts/reproduce.sh
    display(IPImage("results/scale_sweep.png", width=800))
    display(IPImage("results/backend_comparison.png", width=900))
else:
    print("Skipped (set RUN_FULL_REPRODUCTION = True in Settings).")

In [ ]:
import shutil, torch, difflib
manifest = pathlib.Path("outputs/reproduce/checksums.sha256")
if RUN_FULL_REPRODUCTION and manifest.exists() and SAVE_TO_DRIVE:
    gpu = torch.cuda.get_device_name(0).replace(" ", "_") if torch.cuda.is_available() else "cpu"
    ref = pathlib.Path(DRIVE_DIR) / f"checksums_{gpu}_torch{torch.__version__.split('+')[0]}.sha256"
    if ref.exists():
        a = [l for l in ref.read_text().splitlines() if not l.startswith("#")]
        b = [l for l in manifest.read_text().splitlines() if not l.startswith("#")]
        changed = [l for l in difflib.unified_diff(a, b, lineterm="", n=0)
                   if l.startswith("+") and not l.startswith("+++")]
        print(f"Compared with {ref.name}: {len(b) - len(changed)}/{len(b)} files identical.")
        for l in changed[:20]:
            print("  differs:", l.split()[-1])
    else:
        shutil.copy(manifest, ref)
        print(f"No reference yet for this GPU/torch: saved {ref.name}. Re-run later to compare.")
else:
    print("Nothing to compare (needs RUN_FULL_REPRODUCTION and SAVE_TO_DRIVE).")

In [ ]:
# Keep the figures and raw outputs after the Colab VM is recycled.
if SAVE_TO_DRIVE:
    !mkdir -p {DRIVE_DIR}/results && cp -r results/. {DRIVE_DIR}/results/
    !for d in outputs/hero_repro outputs/reproduce; do [ -d "$d" ] && cp -r "$d" {DRIVE_DIR}/; done
    print("Copied to", DRIVE_DIR)

## 6 · *(optional)* Launch the Gradio app

Prints a public `https://….gradio.live` link (valid ~72 h). The cell keeps running while the
app is up; press ■ (stop) to shut it down.

In [ ]:
%cd {REPO_DIR}
if LAUNCH_GRADIO_APP:
    !python app/app.py --share
else:
    print("Skipped (set LAUNCH_GRADIO_APP = True in Settings).")

## Determinism caveats

* **Same GPU model + same library versions → identical images** is the expectation:
  the initial latent comes from a seeded `torch.Generator`, and `set_seed` enables cuDNN
  deterministic mode. The manifest check in step 5 tests exactly this.
* **Different GPU (T4 vs P100/A100) or a different torch/CUDA/diffusers version** changes
  floating-point kernels; fp16 rounding differences accumulate over 30 denoising steps, so
  images are *visually* very close but not bit-identical. Compare those runs by eye or with
  the metrics, not by hash.
* **SDXL** runs with CPU offload, so its noise generator lives on the CPU while SD 1.5's
  lives on the GPU. Both are reproducible, but the same seed does not mean the same noise
  across the two backends.
* The neural annotators (HED/PiDiNet) and CLIP used by `analyze.py` are deterministic in
  eval mode, but their scores inherit any image differences from above.
* `runtime_sec`, peak VRAM, `metrics.csv` and the runtime panel of `backend_comparison.png`
  are measurements, so they change on every run and are left out of the manifest.
* Colab updates its base image (torch, CUDA) over time; step 2 records the versions used,
  and the manifest reference on Drive is keyed by GPU model and torch version.